# 按概念过滤图片：四个本地模型对照

同一批筛选前原图：OK手势24、玻璃棒63、白花芍药27。相同提示词、概念定义、图像字节，4图一批。只比较这一算子，不重跑正文清洗或联合提炼。

独立看图参考是 Codex 初审，**不是专家真值**；物种/材质无法确认的图片单列待定。原始模型判断与现有解析器输出分别显示。组合结果是已有独立判断的离线回放，调用数是按原批次估计，未实测级联时延。

Qwen使用1并发，Gemma使用2并发；临时服务与原服务执行配置不同，耗时不作公平吞吐排名。

表中模型识别统计使用原始语义；原算子输出和修正空/null限制字段后的输出分别保留。A3B仍有16张、Gemma26仍有30张缺字段等协议问题，不当作成功通过；推荐组合两模型在字段修正后均无协议失败。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
RUN = ROOT / 'state/curation/v4/image_filter_models_v1'
import pandas as pd
from IPython.display import display, Markdown
report = json.loads((RUN / 'comparison.json').read_text())
rows = []
for m in report['models']:
    rows.append({'模型': m['model'], '请求数': m['calls'], '并发数': m['concurrency'], '该轮耗时秒（配置不同）': m['evaluation_wall_s'], '字段未通过（原→修正）': str(m['protocol_invalid_images'])+'→'+str(m.get('revalidated_invalid_images', '?')), **m['raw_semantics']})
display(pd.DataFrame(rows).rename(columns={'correct_keep':'相关图保留/47', 'false_exclude':'初审相关图排除*', 'positive_pending':'相关图待定', 'false_keep':'无关图误收', 'correct_exclude':'无关图排除/44', 'negative_pending':'无关图待定', 'uncertain_kept':'身份未定图保留', 'uncertain_excluded':'身份未定图排除', 'uncertain_pending':'身份未定图待定/23'}))
conclusion = RUN / 'conclusion.json'
if conclusion.exists(): display(Markdown(json.loads(conclusion.read_text())['notebook_text']))

## 模型组合

agreement：两者相同才确定，分歧待定。pending_cascade：第一模型仅在待定时交给第二模型。confirm_keep：复核第一模型的入选图，分歧待定。待定不等于已判无关，但当前不会进入联合提炼。

In [ ]:
combos = []
for c in report['combinations']:
    combos.append({'先判':c['first'], '复核':c['second'], '策略':c['policy'], '预计追加批次':c['estimated_second_batches'], **c['raw_semantics']})
pd.set_option('display.max_rows', 50)
display(pd.DataFrame(combos))

## 看原图和逐模型判断

修改 `CONCEPT`、`DISAGREEMENTS_ONLY`、`LIMIT`、`OFFSET` 后运行此 cell。`LIMIT=None` 展示全部；只读已完成结果，不发模型请求。展开“实际观察／解析结果”能看出识图错误和字段协议问题的区别。

默认并排看推荐组合的两个模型及组合判断；`MODELS=None` 可展开四模型。组合来自保存响应的确定性回放。

In [ ]:
from curation.v4.image_filter_review import show_images
CONCEPT = '玻璃棒'  # None / 'OK手势' / '玻璃棒' / '白花芍药'
DISAGREEMENTS_ONLY = True
LIMIT = 20
OFFSET = 0
MODELS = ['qwen3.8-27b', 'gemma-4-31b-it']  # None 展示四模型
show_images(RUN, CONCEPT, DISAGREEMENTS_ONLY, LIMIT, OFFSET, model_names=MODELS)

## 本次实际冻结的提示词及输入范围

In [ ]:
import yaml
manifest = json.loads((RUN / 'session/qwen3.8-27b/manifest.json').read_text())
print(yaml.safe_load(manifest['prompt'])['prompts']['select_images']['template'])
display(json.loads((RUN / 'input_manifest.json').read_text()))
print('每个模型完整请求、响应、提示词快照：', RUN / 'session')

## 初审分歧的放大复查

初审不是专家金标准。以下两张图在事后放大复查中出现身份/形态歧义，**原始标签和主表计数不改写**，另提供剔除这两项的敏感性结果；不能据此制造“零错误金标准成绩”。

In [ ]:
notes = report.get('post_review_notes', {})
display(pd.DataFrame(notes.get('items', [])))
display(pd.DataFrame(report.get('sensitivity_without_disputed_references', {})).T)
from IPython.display import Image as DisplayImage
inventory = [json.loads(line) for line in (RUN / 'inventory.jsonl').read_text().splitlines()]
for row in inventory:
    if row['sample_id'] in {'0-16', '1-01'}:
        display(Markdown(f"**{row['sample_id']} · {row['image_id']}**"))
        display(DisplayImage(filename=row['path'], width=450))